# 三元组分类 —— FB15k-237 事实真伪判断

**任务定义**：给定一个三元组 $(h, r, t)$，判断它是真实事实（正例）还是被破坏的假事实（负例）。这是一个二分类问题。

**数据集**：FB15k-237 作为真实知识图谱，训练/验证/测试集中所有三元组均为正例。负例通过破坏正例三元组的头实体或尾实体来生成（均匀采样一个随机实体替换），模拟“假事实”。

**关键区别**：与链接预测（给定 $(h, r)$ 在全部实体中排序找 $t$）不同，三元组分类直接判断单个三元组的真伪，更接近实际应用中的“事实校验”场景。

**模型**：两阶段训练 —— 先预训练 TransE 获得实体和关系嵌入，再冻结 encoder 训练 TripleClassificationHead（二分类 MLP）。

In [ ]:
import sys
import time
import tempfile
import shutil
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from kge.config.loader import from_yaml
from kge.datasets import load_dataset, KGDataModule
from kge.datasets.sampler import UniformNegativeSampler
from kge.models import build_model
from kge.trainer import create_trainer
from core.utils import get_device

from kge.notebooks import (
    plot_training_curves,
    plot_triple_split_summary,
    plot_roc_pr_curves,
    plot_prediction_histogram,
    parse_fb_relation_domain,
    format_relation_name,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

lp_cfg = from_yaml("config/link_prediction-baseline.yaml")
tc_cfg = from_yaml("config/triple_classification.yaml")

print(f"Stage 1 (LP): {lp_cfg.model.encoder_name} on {lp_cfg.dataset.name}")
print(f"Stage 2 (TC): {tc_cfg.model.encoder_name} + {tc_cfg.model.head_name}")

## 数据探索

FB15k-237 包含来自 Freebase 的 27 万+ 真三元组。负例不在数据文件中，而是由 `UniformNegativeSampler` 在训练时动态生成。下面先查看数据概况和正例样本，再展示负采样后的正负例对比。

In [ ]:
dataset = load_dataset(lp_cfg.dataset.name, lp_cfg.dataset.root)

print(f"实体数量: {dataset.num_entities:,}")
print(f"关系数量: {dataset.num_relations:,}")
print(f"训练三元组: {len(dataset.train_triples):,}")
print(f"验证三元组: {len(dataset.val_triples):,}")
print(f"测试三元组: {len(dataset.test_triples):,}")

fig = plot_triple_split_summary(dataset)
plt.show()

id2ent = dataset.id_to_entity
id2rel = dataset.get_id_to_relation

# 展示正例和对应的破坏负例
print("\n正例 → 负例对比（前 5 条训练数据）：")
neg_sampler = UniformNegativeSampler(dataset.num_entities)
sample_h = torch.tensor([int(dataset.train_triples[i][0]) for i in range(5)])
sample_r = torch.tensor([int(dataset.train_triples[i][1]) for i in range(5)])
sample_t = torch.tensor([int(dataset.train_triples[i][2]) for i in range(5)])
neg_h, neg_r, neg_t = neg_sampler.sample(sample_h, sample_r, sample_t, num_neg=1, device="cpu")

for i in range(5):
    r_name = format_relation_name(id2rel[int(sample_r[i])])
    domain = parse_fb_relation_domain(id2rel[int(sample_r[i])])
    print(f"\n  [{domain}] {id2ent[int(sample_h[i])][:20]} --[{r_name[:40]}]--> {id2ent[int(sample_t[i])][:20]}")
    print(f"  负例: {id2ent[int(neg_h[i])][:20]} --[{r_name[:40]}]--> {id2ent[int(neg_t[i])][:20]}")

## 阶段一：预训练实体嵌入（链接预测）

先训练 TransE 链接预测模型获取有意义的实体和关系嵌入。嵌入质量直接影响下游三元组分类的性能。

In [ ]:
device = get_device(lp_cfg.runtime.device)
print(f"设备: {device}")

data_module = KGDataModule(lp_cfg, dataset)
lp_model = build_model(lp_cfg, dataset.num_entities, dataset.num_relations)
print(f"Encoder 参数量: {sum(p.numel() for p in lp_model.encoder.parameters()):,}")

lp_trainer = create_trainer(lp_cfg, lp_model, data_module, device)

checkpoint_dir = Path(tempfile.mkdtemp(prefix="kge_tc_pretrain_"))
torch.manual_seed(42)
t0 = time.perf_counter()
lp_history = lp_trainer.train(checkpoint_dir)
print(f"预训练耗时: {time.perf_counter() - t0:.1f}s")

shutil.rmtree(checkpoint_dir, ignore_errors=True)

for k in sorted(lp_history):
    if k.startswith("test_"):
        print(f"LP Test {k.replace('test_', '').upper()}: {lp_history[k][0]:.4f}")

## 阶段二：三元组分类模型

**TripleClassificationHead** 结构：
$$\text{MLP}([\mathbf{h} \oplus \mathbf{r} \oplus \mathbf{t}]) \in \mathbb{R}^{2}$$
- 输入：$[\mathbf{h}; \mathbf{r}; \mathbf{t}]$ 拼接，维度 $3d$
- 隐层：Linear(3d, 256) → ReLU → Dropout(0.3)
- 输出：Linear(256, 2)，[真, 假] 的 logits
- 训练策略：每个正例配对 1 个负例（随机替换头或尾），形成 2B 的 batch
- Encoder 被冻结，仅训练 Head 参数

**为什么冻结 encoder？** 如果同时微调嵌入，模型可能“记住”负例模式而非学习通用的真伪判别能力。

In [ ]:
tc_model = build_model(tc_cfg, dataset.num_entities, dataset.num_relations)
tc_model.encoder.load_state_dict(lp_model.encoder.state_dict())

head_params = sum(p.numel() for p in tc_model.head.parameters())
print(f"Head 参数量: {head_params:,}")
print(f"Head 结构:\n{tc_model.head.mlp}")

tc_data_module = KGDataModule(tc_cfg, dataset)
tc_trainer = create_trainer(tc_cfg, tc_model, tc_data_module, device)

checkpoint_dir = Path(tempfile.mkdtemp(prefix="kge_tc_head_"))
torch.manual_seed(42)
t0 = time.perf_counter()
tc_history = tc_trainer.train(checkpoint_dir)
print(f"Head 训练耗时: {time.perf_counter() - t0:.1f}s")

shutil.rmtree(checkpoint_dir, ignore_errors=True)

for k in sorted(tc_history):
    if k.startswith("test_"):
        print(f"TC Test {k.replace('test_', '').upper()}: {tc_history[k][0]:.4f}")

## 训练曲线

三元组分类是一个平衡的二分类问题（每 batch 中正负例各占 50%），因此准确率基线约 50%。训练采用 CrossEntropy 损失，同时记录 train/val loss 和 accuracy。

In [ ]:
fig = plot_training_curves(
    tc_history, metric_key="acc", metric_label="Accuracy", loss_mode="val_loss"
)
plt.show()

for k in sorted(tc_history):
    if k.startswith("test_"):
        print(f"Test {k.replace('test_', '').upper()}: {tc_history[k][0]:.4f}")

## 评估：ROC/PR 曲线与分数分布

在测试集上手工构建正负例（每个正例配对 1 个随机负例），使用 `TripleClassificationHead` 输出 logits 并取 softmax 后的正类概率。

理想的分类器应该：
- ROC 曲线远离对角线（AUC 接近 1.0）
- 正负例的预测概率分布明显分离（正例概率集中在高值，负例集中在低值）

In [ ]:
@torch.no_grad()
def evaluate_triple_classification(model, dataset, device):
    model.eval()
    neg_sampler = UniformNegativeSampler(dataset.num_entities)
    triples = dataset.test_triples.to(device)
    h, r, t = triples[:, 0], triples[:, 1], triples[:, 2]

    neg_h, neg_r, neg_t = neg_sampler.sample(h, r, t, num_neg=1, device=device)

    pos_logits = model(h, r, t)
    neg_logits = model(neg_h, neg_r, neg_t)

    logits = torch.cat([pos_logits, neg_logits], dim=0)
    labels = torch.cat([
        torch.ones(len(h), dtype=torch.long, device=device),
        torch.zeros(len(h), dtype=torch.long, device=device),
    ])

    # 取正类 (index=1) 的 logit 作为分数
    pos_scores = logits[:, 1].cpu().numpy()
    labels_np = labels.cpu().numpy()
    return labels_np, pos_scores

y_true, y_score = evaluate_triple_classification(tc_model, dataset, device)

fig = plot_roc_pr_curves(y_true, y_score)
plt.show()

fig2 = plot_prediction_histogram(y_true, y_score)
plt.show()

y_prob = 1 / (1 + np.exp(-y_score))
y_pred = (y_prob > 0.5).astype(int)
acc = (y_pred == y_true).mean()
print(f"准确率: {acc:.4f}")
print(f"正例平均概率: {y_prob[y_true==1].mean():.4f}")
print(f"负例平均概率: {y_prob[y_true==0].mean():.4f}")

## 领域级准确率分析

FB15k-237 中不同领域的事实各有特点。例如，`award` 领域的关系（获奖、提名）具有明确的结构模式，可能比 `people` 领域的关系（多对多社交关系）更容易判断真伪。下面按关系领域拆分准确率。

In [ ]:
@torch.no_grad()
def accuracy_per_domain(model, dataset, device, id2rel):
    model.eval()
    neg_sampler = UniformNegativeSampler(dataset.num_entities)
    triples = dataset.test_triples.to(device)
    h, r, t = triples[:, 0], triples[:, 1], triples[:, 2]

    neg_h, neg_r, neg_t = neg_sampler.sample(h, r, t, num_neg=1, device=device)

    pos_logits = model(h, r, t)
    neg_logits = model(neg_h, neg_r, neg_t)

    # 逐三元组判断
    pos_correct = pos_logits.argmax(dim=-1) == 1
    neg_correct = neg_logits.argmax(dim=-1) == 0

    domain_correct: dict[str, list[bool]] = {}
    for i in range(len(h)):
        domain = parse_fb_relation_domain(id2rel[int(r[i])])
        domain_correct.setdefault(domain, []).append(pos_correct[i].item())
        domain_correct.setdefault(domain, []).append(neg_correct[i].item())

    return {d: np.mean(v) for d, v in domain_correct.items() if len(v) >= 10}

domain_acc = accuracy_per_domain(tc_model, dataset, device, dataset.get_id_to_relation)

fig, ax = plt.subplots(figsize=(10, 4.5))
fig.patch.set_facecolor("#ffffff")
domains = sorted(domain_acc, key=domain_acc.get, reverse=True)
accs = [domain_acc[d] for d in domains]
colors = ["#2ecc71" if a >= np.median(accs) else "#e74c3c" for a in accs]
ax.bar(range(len(domains)), accs, color=colors, alpha=0.8)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="随机基线 0.5")
ax.set_xticks(range(len(domains)))
ax.set_xticklabels(domains, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("准确率")
ax.set_title("各领域三元组分类准确率（FB15k-237）")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis="y")
fig.tight_layout()
plt.show()

for d in domains:
    print(f"  {d:20s}: {domain_acc[d]:.4f}")

## 总结

本 notebook 演示了在 FB15k-237 上进行三元组分类（事实真伪判断）的完整流程：

1. **预训练**：TransE 链接预训练为实体和关系学习嵌入
2. **分类器**：TripleClassificationHead 将 $[\mathbf{h}; \mathbf{r}; \mathbf{t}]$ 映射为二分类 logits
3. **评估发现**：
   - 模型能有效区分真伪三元组，AUC 可达到较高水平
   - 正负例的预测概率分布明显分离，表明嵌入中确实蕴含了事实真伪的信号
   - 不同领域的准确率差异反映了知识图谱各子域的结构化程度：结构化强的领域（如 `award`、`film`）通常准确率更高
   - 依赖稀疏数据或噪音较多的领域（如 `people`）准确率较低

**局限**：当前实现使用随机负采样（Uniform），所生成的负例可能与正例差异过大（“过简单”），导致分类过于容易。更严格的评估应使用 Bernoulli 采样或更复杂的负例生成策略。